In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from unpywall import Unpywall
from unpywall.utils import UnpywallCredentials
import pandas as pd
import numpy as np
import re
import time

https://www.biorxiv.org/search/title%3Aprotein%252C%2Bpeptide%252C%2Bnanobody%252C%2Bantibody%252C%2Benzyme%252C%2BTCR%20title_flags%3Amatch-any%20jcode%3Abiorxiv%20limit_from%3A2025-05-01%20limit_to%3A2025-05-05%20numresults%3A75%20sort%3Apublication-date%20direction%3Adescending%20format_result%3Astandard

Another option would be https://github.com/jannisborn/paperscraper, but would take longer likely as it downloads everything.
But given how slow soup is it could actually be quicker

Unpaywall does not have abstracts

In [4]:
from datetime import datetime, timedelta
def next_day(date):
    date_obj = datetime.strptime(date, "%Y/%m/%d")
    next_day = date_obj + timedelta(days=1)
    next_day = next_day.strftime("%Y/%m/%d")

    return next_day


def list_dates(start_date, end_date):
    """
    List all dates between the start and end date, inclusive.

    Parameters:
    - start_date: Start date in the format 'YYYY/MM/DD'
    - end_date: End date in the format 'YYYY/MM/DD'

    Returns:
    - List of dates as strings in 'YYYY/MM/DD' format
    """
    # Convert string dates to datetime objects
    start = datetime.strptime(start_date, '%Y/%m/%d')
    end = datetime.strptime(end_date, '%Y/%m/%d')

    # Create a list of dates
    date_list = []
    current_date = start

    while current_date <= end:
        date_list.append(current_date.strftime('%Y/%m/%d'))
        current_date += timedelta(days=1)  # Increment by one day

    return date_list


In [5]:
def get_url(url):
    # Send a request to the bioRxiv website
    response = requests.get(url)
    
    # Check if the request was successful
    if response.status_code != 200:
        print("Failed to retrieve data")
    
    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

def construct_biorxiv_url(date):
    date_from = date
    date_to = next_day(date)

    # Format dates for arXiv API (YYYYMMDD format)
    date_from_formatted = date_from.replace('/', '-')
    date_to_formatted = date_to.replace('/', '-')
    
    query='title%3Aprotein%252C%2Bpeptide%252C%2Bnanobody%252C%2Bantibody%252C%2Benzyme%252C%2BTCR%20'+\
    'title_flags%3Amatch-any%20jcode%3Abiorxiv%20'+\
    f'limit_from%3A{date_from_formatted}%20limit_to%3A{date_to_formatted}'+\
    '%20numresults%3A75%20sort%3Apublication-date%20direction%3Adescending%20format_result%3Astandard'
    
    url = f"https://www.biorxiv.org/search/{query}"

    return url

def fetch_biorxiv_results(url):

    soup = get_url(url)
    publications = soup.find_all('div', class_='highwire-article-citation')

    results = []

    for pub in publications:
        doi=pub.find('span', class_='highwire-cite-metadata-doi'
                               ).get_text(strip=True).split('https://doi.org/')[1]
        date='-'.join(doi.split('/')[1].split('.')[:3])
        soup=get_url('https://doi.org/'+doi)
        title=soup.find_all('title')[0].get_text(strip=True).split(' | ')[0]
        abstract=soup.find_all('div', id='abstract-1')[0].find_all('p')[0].get_text(strip=True)
        results.append({
            'title':title,
            'date':date,
            'doi':doi,
            'abstract':abstract
        })

    return results

def run_biorxiv_search(start_date, end_date):
    print('Retrieving biorxiv')
    all_results = []

    dates_list = list_dates(start_date, end_date)

    for date in dates_list:
        # Construct the API query URL
        url = construct_biorxiv_url(date)

        # Fetch the results
        print(f"Fetching results from bioRxiv API for date {date}...")
        results = fetch_biorxiv_results(url)

        # Print the results
        print(f"\nFound {len(results)} articles for date {date}\n")

        for article in results:
            article['search_date'] = date.replace('/', '-')
            all_results.append(article)

    return pd.DataFrame(all_results)


In [6]:
run_biorxiv_search('2025/05/01', '2025/05/04')

Retrieving biorxiv
Fetching results from bioRxiv API for date 2025/05/01...


KeyboardInterrupt: 

Biorxiv is very slow

In [19]:
%%time
_=get_url(construct_biorxiv_url('2025/05/05'))

CPU times: user 81.1 ms, sys: 7.32 ms, total: 88.4 ms
Wall time: 7.36 s


In [26]:
%%time
_=get_url('https://www.biorxiv.org/content/10.1101/2025.04.28.650887v3')

CPU times: user 70.7 ms, sys: 7.11 ms, total: 77.8 ms
Wall time: 6.38 s


In [25]:
%%time
_=requests.get('https://www.biorxiv.org/content/10.1101/2025.04.28.650887v3')

CPU times: user 24.3 ms, sys: 7.22 ms, total: 31.5 ms
Wall time: 8.76 s


## biorxiv api

In [30]:
def filter_result(result):
    text=result['title']
    text=text.lower()
    text=text.replace('-',' ')
    # OR within query_sets and AND between them
    queries=[
        ["protein*","peptide*","antibod*","nanobod*","enzyme*","tcr","t cell receptor*"],
        ["learn*","deep*","neural network*","diffusion*","transformer*","flow matching","predict*","generative",
        "embedding*","representation","benchmark*","supervised*","unsupervised*","design*","structure*","model*"],

    ]
    outcomes=[]
    for query_set in queries:
        outcomes.append(any([bool(re.search(query, text)) for query in  query_set]))
    return all(outcomes)

def process_result(result):

    # Parse date - the one in doi is earliest published date but is not present for old publications
    doi_end=result['doi'].split('/')[-1]
    if len(doi_end.split('.'))>3: # Date followed by id
        date='-'.join(doi_end.split('.')[:3])
    else:
        date=result['date']
        
    return {
        'title':result['title'],
        'abstract':result['abstract'],
        'doi':result['doi'],
        'date':date,
    }
    

def fetch_biorxiv_results(date, max_retries=3, retry_delay=5):

    date=date.replace('/', '-')
    
    cursor_max=np.inf
    cursor_curr=0
    results=[]
    bad_status=False
    while cursor_curr<cursor_max and not bad_status:
        for attempt in range(max_retries):
            try:
                api_url=f'https://api.biorxiv.org/details/biorxiv/{date}/{date}/{cursor_curr}/json'
                response = requests.get(api_url, timeout=30)
                
                # Check that response is ok for this day
                if response.json()['messages'][0]['status']!='ok':
                    print('bioRxiv could not retrieve results with status: '+
                          response.json()['messages'][0]['status'])
                    bad_status=True
                    break
    
                # Save filtered results
                results.extend([
                    process_result(res) for res in response.json()["collection"] if filter_result(res)
                ])
    
                # Increase retrieval counter as API retrieves batches of 100
                message=response.json()['messages'][0]
                if cursor_max==np.inf:
                    cursor_max=int(message['total'])
                cursor_curr+=int(message['count'])
    
                # Finish loop if successful
                break
                
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    print(
                        f"Error fetching results (attempt {attempt + 1}/{max_retries}): {e}")
                    print(f"Retrying in {retry_delay} seconds...")
                    time.sleep(retry_delay)
                else:
                    print(
                        f"Failed to fetch results after {max_retries} attempts: {e}")
    
    
    return results       

def run_biorxiv_search(start_date, end_date):
    print('Retrieving biorxiv')
    all_results = []

    dates_list = list_dates(start_date, end_date)

    for date in dates_list:

        # Fetch the results
        print(f"Fetching results from bioRxiv API for date {date}...")
        results = fetch_biorxiv_results(date)

        # Print the results
        print(f"\nFound {len(results)} articles for date {date}\n")

        for article in results:
            article['search_date'] = date.replace('/', '-')
            all_results.append(article)

    return pd.DataFrame(all_results)

In [33]:
papers=run_biorxiv_search('2025/03/13', '2025/03/13')

Retrieving biorxiv
Fetching results from bioRxiv API for date 2025/03/13...

Found 13 articles for date 2025/03/13



In [21]:
date='2025-03-08'
cursor_curr=0
api_url=f'https://api.biorxiv.org/details/biorxiv/{date}/{date}/{cursor_curr}/json'
response = requests.get(api_url, timeout=30)

In [23]:
response.json()['messages']

[{'status': 'no posts found'}]

In [25]:
response.json()['messages'][0]['status']

'no posts found'